# Research Pipeline: Data Loading, Sampling, Preprocessing, and LIME Attribution
This notebook loads SNLI dataset, samples subsets, preprocesses them for attribution, and runs LIME explanations on SNLI examples.

Done for 300 samples of snli without stop words


## PART 1: Setup environment and data

### Import necessary libraries

In [19]:
import os
import re
import json
import time
import torch, torch.nn as nn
import random
import hashlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import pandas as pd
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import warnings
import datetime
import subprocess
from sklearn.metrics import confusion_matrix
import transformers
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords")
matplotlib.use("Agg")

# Silence noisy logs
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()

In [20]:
# === Labels (single source of truth) ===
# SNLI dataset ground-truth indices:
#   0 = entailment, 1 = neutral, 2 = contradiction
SNLI_LABELS = ['entailment', 'neutral', 'contradiction']

# roberta-large-mnli logits/probability indices:
#   0 = contradiction, 1 = neutral, 2 = entailment
MNLI_LABELS = ['contradiction', 'neutral', 'entailment']

# useful maps for when you must compare integers
SNLI_TO_MNLI_IDX = {SNLI_LABELS.index(name): MNLI_LABELS.index(name) for name in SNLI_LABELS}
MNLI_TO_SNLI_IDX = {v: k for k, v in SNLI_TO_MNLI_IDX.items()}

SCHEMA_VERSION = "1.2"   # single source of truth

SEP = "[SEP]"  # single separator
SEP_WITH_SPACES = f" {SEP} "  # canonical form

LIME_SCOPE = "hypothesis"  # options: "pair" (old behavior), "hypothesis"

# Set reproducible seeds
def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    print(f"✅ All seeds set to {seed}")

def get_git_hash():
    """Get current git hash for reproducibility"""
    try:
        git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], 
                                         stderr=subprocess.DEVNULL).decode('ascii').strip()
        return git_hash[:8]  # Short hash
    except Exception:
        return "no-git"

def save_run_config(dirs, **kwargs):
    """Save run configuration for reproducibility"""
    config = {
        "schema_version": SCHEMA_VERSION,
        "timestamp": datetime.datetime.now().isoformat(),
        "git_hash": get_git_hash(),
        "model_name": "roberta-large-mnli",
        "random_seed": 42,
        "num_samples": kwargs.get('num_samples', 50),
        "chunk_size": kwargs.get('chunk_size', 64),
        "lime_num_samples": kwargs.get('lime_num_samples', 1000),
        "lime_num_features": 10,
        "k_tokens": 3,
        "snli_labels": SNLI_LABELS,
        "mnli_labels": MNLI_LABELS,
        "snli_to_mnli_idx": SNLI_TO_MNLI_IDX,
    }
    
    config_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "run_config.json")
    write_json(config_path, config)
    print(f"✅ Run config saved to: {config_path}")
    return config

def compute_hash(obj_or_path):
    """Stable hash for files or in-memory objects"""
    try:
        if isinstance(obj_or_path, str) and os.path.exists(obj_or_path):
            with open(obj_or_path, "rb") as f:
                return hashlib.md5(f.read()).hexdigest()
        payload = _to_py(obj_or_path)
        return hashlib.md5(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    except Exception:
        return f"hash-fallback-{time.time()}"

def should_reuse(path, expected_meta: dict) -> tuple[bool, dict]:
    """Return (reuse?, previous_obj). Reuse if file exists and meta matches."""
    if not file_nonempty(path):
        return False, {}
    try:
        prev = read_json(path)
        meta = prev.get("meta", {})
        for k, v in expected_meta.items():
            if meta.get(k) != v:
                return False, prev
        return True, prev
    except Exception:
        return False, {}

def _to_py(o):
    if isinstance(o, dict):
        return { _to_py(k): _to_py(v) for k, v in o.items() }
    if isinstance(o, (list, tuple)):
        return [ _to_py(v) for v in o ]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o  # fall back

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def file_nonempty(path: str) -> bool:
    return os.path.isfile(path) and os.path.getsize(path) > 0

def read_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(path: str, data):
    data = with_schema(data)
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_py(data), f, indent=2, ensure_ascii=False)

def with_schema(d):
    # attach schema unless the object already has it
    if isinstance(d, dict) and "schema_version" not in d:
        d = {"schema_version": SCHEMA_VERSION, **d}
    return d

def join_pair(premise: str, hypothesis: str) -> str:
    return f"{premise}{SEP_WITH_SPACES}{hypothesis}"

def split_pair(text: str) -> tuple[str, str]:
    idx = text.find(SEP)
    if idx == -1:
        raise ValueError(f"[split_pair] Missing '{SEP}' in: {text[:200]}")
    left = text[:idx].rstrip()
    right = text[idx + len(SEP):].lstrip()
    # ✅ allow empty right (or left) — do not raise
    return left, right

def _normalize_ws(s: str) -> str:
    # collapse all kinds of whitespace, preserve punctuation
    return " ".join(str(s).split())

def _chk(i, r):
    it = r["input_text"]
    # sanity: exactly one SEP
    sep_count = it.count("[SEP]")
    if sep_count != 1:
        return f"id={r['id']} has {sep_count} '[SEP]' tokens"

    p_split, h_split = split_pair(it)  # the same function the pipeline uses
    # normalize both sides before comparing
    p_ok = _normalize_ws(p_split) == _normalize_ws(r["premise"])
    h_ok = _normalize_ws(h_split) == _normalize_ws(r["hypothesis"])

    if not p_ok or not h_ok:
        # only report real content drift (ignore pure whitespace differences)
        if p_ok is False or h_ok is False:
            return f"id={r['id']} mismatch after split (premise_ok={p_ok}, hyp_ok={h_ok})"
    return None

def backfill_all_probabilities_inplace(json_path: str, classifier, schema_version: str = None):
    """If records lack `all_probabilities`, compute from classifier and write back in-place."""
    if not file_nonempty(json_path):
        return False
    raw = read_json(json_path)
    data = raw.get("data", raw) if isinstance(raw, dict) else raw
    changed = False
    for r in data:
        if "all_probabilities" not in r or not isinstance(r["all_probabilities"], dict):
            probs = classifier.predict_proba([r["input_text"]])[0]
            r["all_probabilities"] = {MNLI_LABELS[i]: float(probs[i]) for i in range(len(MNLI_LABELS))}
            changed = True
    if changed:
        payload = {"schema_version": schema_version or SCHEMA_VERSION, "data": data}
        write_json(json_path, payload)
    return changed

def _select_attributions(rec):
    """
    Prefer filtered attributions if present; else fall back to raw.
    Always returns a sorted list of (token, score) pairs (may be empty).
    """
    attrs = rec.get("lime_attributions_filtered")
    if not isinstance(attrs, list) or not attrs:
        attrs = rec.get("lime_attributions", [])
    # keep only (str, float-like)
    out = []
    for it in attrs:
        try:
            w, s = it
            out.append((str(w), float(s)))
        except Exception:
            continue
    # sort by absolute contribution
    out.sort(key=lambda x: abs(x[1]), reverse=True)
    return out


### Set Up Data Paths & Directories

In [21]:
def setup_directories(run_name: str | None = None):
    """Setup all required directories.

    Shared caches live in data/.
    Run-specific *write* artifacts live in data/runs/<run_name>/ if provided,
    otherwise they default to data/ (backward compatible).
    """
    CWD = os.getcwd()
    BASE_DATA_DIR = os.path.join(CWD, "data")

    # Shared, re-usable caches (do NOT change across runs)
    CACHE_DIR = os.path.join(BASE_DATA_DIR, "hf_cache")
    SNLI_LOCAL_DIR = os.path.join(BASE_DATA_DIR, "snli")

    # Run-specific roots for *this* execution
    if run_name:
        RUN_ROOT = os.path.join(BASE_DATA_DIR, "runs", run_name)
    else:
        RUN_ROOT = BASE_DATA_DIR  # backward compat: write into data/

    directories = {
        "DATA_DIR": BASE_DATA_DIR,
        "CACHE_DIR": CACHE_DIR,
        "SNLI_LOCAL_DIR": SNLI_LOCAL_DIR,
        # All WRITE paths go under RUN_ROOT:
        "SAMPLE_SNLI_DIR":    os.path.join(RUN_ROOT, "sampled_snli_data"),
        "PROCESSED_SNLI_DIR": os.path.join(RUN_ROOT, "processed_snli_data"),
        "LIME_OUTPUT_DIR":    os.path.join(RUN_ROOT, "lime_outputs"),
    }

    # Create all directories we may write to
    os.makedirs(directories["SAMPLE_SNLI_DIR"], exist_ok=True)
    os.makedirs(directories["PROCESSED_SNLI_DIR"], exist_ok=True)
    os.makedirs(directories["LIME_OUTPUT_DIR"], exist_ok=True)
    os.makedirs(os.path.join(directories["LIME_OUTPUT_DIR"], "plots"), exist_ok=True)

    # Ensure shared caches exist (harmless if already there)
    os.makedirs(directories["CACHE_DIR"], exist_ok=True)
    os.makedirs(directories["SNLI_LOCAL_DIR"], exist_ok=True)

    print(f"📂 RUN_ROOT: {RUN_ROOT}")
    return directories


### Load Datasets and preview (with Local Cache)
Load SNLI dataset from disk if available, otherwise download and cache them locally.

In [22]:
def load_snli_dataset_fixed(dirs):
    """Load SNLI dataset with proper error handling"""
    print("📥 Loading SNLI dataset...")

    dataset_path = dirs['SNLI_LOCAL_DIR']
    dataset_ready = os.path.exists(os.path.join(dataset_path, "dataset_dict.json"))  # or "state.json"

    if dataset_ready:
        print("Loading from local cache...")
        snli_data = load_from_disk(dataset_path)
    else:
        print("Downloading SNLI dataset...")
        snli_data = load_dataset("snli", cache_dir=dirs['CACHE_DIR'])
        snli_data.save_to_disk(dataset_path)

    # Validate dataset
    if 'train' not in snli_data:
        raise ValueError("SNLI dataset does not contain 'train' split.")
    
    sample = snli_data['train'][0]
    required_fields = ['premise', 'hypothesis', 'label']
    for field in required_fields:
        if field not in sample:
            raise ValueError(f"SNLI dataset missing field: {field}")
    
    print(f"✅ SNLI dataset loaded: {len(snli_data['train'])} training examples")
    print("Sample:", sample)
    
    return snli_data

### Sample 300 Examples from Each Dataset and Save to disk
Randomly sample 300 valid examples from SNLI and CommonsenseQA for fast experimentation.

In [23]:
def sample_snli_dataset_fixed(dataset, dirs, num_samples=300):
    """Ensure we have at least `num_samples` unique, valid items; extend if needed."""
    sample_json_path = os.path.join(dirs['SAMPLE_SNLI_DIR'], "snli_sample.json")

    # start from existing
    existing = []
    if file_nonempty(sample_json_path):
        print(f"⏩ Found existing SNLI sample at {sample_json_path}; loading…")
        raw = read_json(sample_json_path)
        existing = raw.get("data", raw) if isinstance(raw, dict) else raw

    have = { (e["premise"], e["hypothesis"]) for e in existing }
    target = int(num_samples)

    if len(existing) >= target:
        print(f"✅ Already have {len(existing)} samples (>= {target}); reusing.")
        return existing

    print(f"🎯 Need {target} samples; currently {len(existing)}. Extending…")
    # collect additional, unique, valid examples
    valid_indices = [i for i, ex in enumerate(dataset['train']) if ex['label'] != -1]
    random.seed(42)
    random.shuffle(valid_indices)

    next_id = max([e.get("id", -1) for e in existing] + [-1]) + 1
    for idx in valid_indices:
        if len(existing) >= target:
            break
        ex = dataset['train'][idx]
        key = (ex['premise'], ex['hypothesis'])
        if key in have:
            continue
        existing.append({
            "id": next_id,
            "premise": ex["premise"],
            "hypothesis": ex["hypothesis"],
            "label": int(ex["label"]),
            "label_name": SNLI_LABELS[int(ex["label"])]
        })
        have.add(key); next_id += 1

    write_json(sample_json_path, with_schema({"data": existing}))
    print(f"✅ Saved {len(existing)} samples to: {sample_json_path}")
    return existing

### Preprocess Sampled Data for Attribution
Convert SNLI samples into model-ready format and save for later use in LIME/SHAP or other explainers.

In [24]:
def preprocess_snli_for_roberta(snli_sample_data, dirs):
    """
    Preprocess SNLI data specifically for RoBERTa-MNLI format
    Preprocess once; if processed file exists, load and return.
    """
    processed_path = os.path.join(dirs['PROCESSED_SNLI_DIR'], "processed_snli.json")
    if file_nonempty(processed_path):
        print(f"⏩ Found existing processed SNLI at {processed_path}; loading…")
        return read_json(processed_path)

    print("🔄 Preprocessing SNLI data for RoBERTa-MNLI...")
    
    processed_snli = []
    for ex in snli_sample_data:
        # RoBERTa format: premise </sep></sep> hypothesis
        # But tokenizer handles this automatically, so we just use: premise <sep> hypothesis
        text = join_pair(ex['premise'], ex['hypothesis'])
        
        processed_entry = {
            "id": ex["id"],
            "input_text": text,
            "premise": ex["premise"],
            "hypothesis": ex["hypothesis"],
            "label": ex["label"],
            "label_name": ex["label_name"],
            "dataset": "snli"
        }
        processed_snli.append(processed_entry)

    changed = 0
    for r in processed_snli:
        p_split, h_split = split_pair(r["input_text"])
        # compare with whitespace normalization only
        if _normalize_ws(p_split) != _normalize_ws(r["premise"]) or _normalize_ws(h_split) != _normalize_ws(r["hypothesis"]):
            # auto-correct to what we actually use downstream
            r["premise"] = p_split
            r["hypothesis"] = h_split
            changed += 1

    if changed:
        print(f"🔧 Auto-corrected premise/hypothesis on {changed} records to match input_text split")
    
    # Save processed data
    write_json(processed_path, processed_snli)
    
    print(f"✅ Saved {len(processed_snli)} processed examples to: {processed_path}")
    
    return processed_snli

## PART 2: ROBERTA-MNLI MODEL SETUP


In [25]:
class RoBERTaMNLIClassifier:
    """Proper RoBERTa-MNLI classifier for LIME explanations"""
    
    def __init__(self, model_name="roberta-large-mnli", use_fp16=True):
        print(f"🤖 Loading {model_name} model...")
        
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
        # Load model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        if use_fp16 and self.device.type == "cuda":
            self.model.half()
        self.model.to(self.device)
        self.model.eval()
        
        # Label mapping for MNLI (RoBERTa uses different order than SNLI)
        self.label_mapping = {i: name for i, name in enumerate(MNLI_LABELS)}
        
        print("✅ Model loaded successfully!")
        self._test_model()
    
    def _test_model(self):
        """Test model with a simple example"""
        print("🧪 Testing model...")
        
        test_premise = "The cat is sleeping on the couch."
        test_hypothesis = "The cat is awake."
        test_text = join_pair(test_premise, test_hypothesis)
        
        probs = self.predict_proba([test_text])[0]
        predicted_label = int(np.argmax(probs))
        confidence = float(np.max(probs))
        
        print(f"Test input: '{test_premise}' vs '{test_hypothesis}'")
        print(f"Probabilities: {probs}")
        print(f"Predicted: {self.label_mapping[predicted_label]} (confidence: {confidence:.4f})")
        
        # Should predict contradiction with high confidence
        if predicted_label == 0 and confidence > 0.7:
            print("✅ Model test passed!")
        else:
            print("⚠️ Model test results seem unusual, but proceeding...")
    
    def predict_proba(self, texts):
        """Predict probabilities for LIME (batch processing)"""
        if isinstance(texts, str):
            texts = [texts]
        else:
            # normalize numpy/object arrays to a flat list of strings
            texts = [str(x) for x in np.array(texts, dtype=object).ravel().tolist()]

        pairs = [split_pair(t) for t in texts]  # strict [SEP] split
        premises = [p for p, _ in pairs]
        hyps = [h for _, h in pairs]

        inputs = self.tokenizer(
            premises,
            hyps,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        # Ensure 2D shape
        if probs.ndim == 1:
            probs = probs.reshape(1, -1)

        return probs

    def predict_single(self, text):
        """Get single prediction with details"""
        probs = self.predict_proba([text])[0]
        predicted_label = int(np.argmax(probs))
        
        return {
            'probabilities': probs,
            'predicted_label': predicted_label,
            'predicted_class': self.label_mapping[predicted_label],
            'confidence': float(np.max(probs)),
            'all_probs': {self.label_mapping[i]: probs[i] for i in range(len(probs))}
        }
    
    def predict_logits(self, texts):
        pairs = [split_pair(t) for t in texts]
        premises = [p for p, _ in pairs]
        hyps = [h for _, h in pairs]

        enc = self.tokenizer(
            premises, hyps,
            padding=True, truncation=True,
            max_length=256,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(**enc).logits
        return logits.detach().cpu().numpy()

## PART 3: LIME EXPLANATIONS WITH PROPER EVALUATION


In [26]:
def explain_hypothesis_lime(premise: str, hypothesis: str, clf_predict_proba, *,
                            num_features=10, num_samples=5000, split_expression=r"\s+"):
    """
    Runs LIME on *hypothesis only*. The predictor stitches back premise + hypothesis variants.
    """
    explainer = LimeTextExplainer(split_expression=split_expression)
    def hyp_predict(hyps):
        stitched = [join_pair(premise, h) for h in hyps]
        return clf_predict_proba(stitched)  # returns probs in MNLI order
    exp = explainer.explain_instance(hypothesis, hyp_predict, num_features=num_features, num_samples=num_samples)
    return exp.as_list(), float(exp.score)  # [(token, score), ...], R^2-like score

def generate_lime_explanations(classifier, processed_snli, dirs, num_examples=50,
                               lime_num_samples=1000, chunk_size=64,
                               force_rebuild=False, save_every=10):
    """
    Generate LIME explanations with proper RoBERTa integration.
    Resumable: keeps existing items, appends new ones until num_examples.
    """
    lime_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_roberta.json")
    
    errors = []
    for i, r in enumerate(processed_snli):
        e = _chk(i, r)
        if e: errors.append(e)

    print("✅ processed_snli.json validation passed" if not errors else f"❌ issues: {len(errors)}")
    if errors:
        print("\n".join(errors[:10]))

    # Load any existing data (resumable)
    existing = []
    if file_nonempty(lime_path) and not force_rebuild:
        try:
            raw = read_json(lime_path)
            existing = raw.get("data", raw) if isinstance(raw, dict) else raw
            print(f"⏩ Found existing LIME explanations at {lime_path} with {len(existing)} items; will resume.")
        except Exception as e:
            print(f"⚠ Failed to read existing LIME file ({e}). Will regenerate.")

    processed_ids = {int(r["id"]) for r in existing if "id" in r}
    target_total = int(num_examples)
    to_go = max(0, target_total - len(existing))
    if to_go == 0:
        print("⏩ Already have requested number of LIME items; returning cached.")
        return existing[:target_total]

    print(f"🔍 Generating LIME explanations for {to_go} new examples (target={target_total})...")
    print(f"   LIME samples: {lime_num_samples}, Chunk size: {chunk_size}")

    new_items = []
    seen = 0

    for ex in tqdm(processed_snli, desc="LIME Explanations"):
        if len(existing) + len(new_items) >= target_total:
            break
        if int(ex["id"]) in processed_ids:
            continue

        try:
            text = ex["input_text"]
            premise, hypothesis = split_pair(text)
            t0 = time.perf_counter()
            prediction = classifier.predict_single(text)
            explanation_pairs, explanation_score = explain_hypothesis_lime(
                premise=premise,
                hypothesis=hypothesis,
                clf_predict_proba=classifier.predict_proba,
                num_features=10,
                num_samples=int(lime_num_samples),
                split_expression=r"\s+"
            )
            
            assert isinstance(explanation_pairs, (list, tuple)), "LIME attributions must be a list of (token, score)"
            
            latency = time.perf_counter() - t0
            item = {
                "id": int(ex["id"]),
                "premise": ex["premise"],
                "hypothesis": ex["hypothesis"],
                "input_text": text,
                "true_label": int(ex["label"]),
                "true_label_name": ex["label_name"],
                "predicted_label": int(prediction["predicted_label"]),
                "predicted_class": prediction["predicted_class"],
                "confidence": float(prediction["confidence"]),
                "lime_attributions": [(w, float(s)) for (w, s) in explanation_pairs],
                "lime_score": float(explanation_score),
                "lime_runtime_sec": float(latency),
                "all_probabilities": prediction["all_probs"],
            }
            new_items.append(item)
            seen += 1

            # checkpoint
            if seen % save_every == 0:
                merged = {"schema_version": SCHEMA_VERSION, "data": existing + new_items}
                write_json(lime_path, merged)
                print(f"💾 Checkpoint: saved {len(existing)+len(new_items)}/{target_total} items")

        except Exception as e:
            print(f"❌ Error on id={ex.get('id')}: {e}")

    merged = {"schema_version": SCHEMA_VERSION, "data": existing + new_items}
    write_json(lime_path, merged)
    print(f"✅ Generated {len(new_items)} new LIME explanations (total={len(existing)+len(new_items)})")
    print(f"✅ Saved to: {lime_path}")
    return (existing + new_items)[:target_total]

In [27]:
def filter_stopwords_from_lime(lime_results, dirs):
    """Remove stopwords from LIME attributions for cleaner analysis"""
    if not lime_results:
        print("⏩ No LIME results; skipping stopword filtering.")
        return []
    print("🧹 Filtering stopwords from LIME attributions...")

    # identify the source by hashing ids + raw attributions (light but stable)
    src_hash = compute_hash([(r["id"], r.get("lime_attributions", [])) for r in lime_results])

    filtered_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_filtered.json")
    reuse, prev = should_reuse(filtered_path, {"source": "lime_raw", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing filtered results from: {filtered_path}")
        return prev.get("data", prev)
    
    # Setup stopwords
    stopwords_set = set(stopwords.words('english'))
    additional_stopwords = {"[SEP]", "[CLS]", "[PAD]", "[UNK]", "[MASK]", "SEP"}
    stopwords_set.update(additional_stopwords)
    
    filtered_results = []
    for result in lime_results:
        atts = result.get("lime_attributions", [])
        filtered_atts = [(t, s) for t, s in atts if t.lower() not in stopwords_set and len(t.strip()) > 1]
        out = result.copy()
        out["lime_attributions_filtered"] = filtered_atts
        out["original_attribution_count"] = len(atts)
        out["filtered_attribution_count"] = len(filtered_atts)
        filtered_results.append(out)
    
    # Save filtered results with schema
    filtered_data_with_schema = {
        "schema_version": SCHEMA_VERSION,
        "meta": {"source": "lime_raw", "source_hash": src_hash},
        "data": filtered_results
    }
    write_json(filtered_path, filtered_data_with_schema)
    
    print(f"✅ Filtered results saved to: {filtered_path}")
    print(f"Average attribution reduction: {np.mean([r['original_attribution_count'] - r['filtered_attribution_count'] for r in filtered_results]):.1f} tokens")
    
    return filtered_results

## PART 4: EVALUATION METRICS


In [28]:
# --- Canonical per-example metric computation ---
def compute_example_metrics(input_text: str,
                            lime_attributions: list[tuple[str, float]],
                            clf,
                            k: int) -> dict:
    """
    input_text: "premise [SEP] hypothesis"
    lime_attributions: [(token, score), ...] from hypothesis-only LIME
    clf: has predict_proba([...])
    """

    premise, hypothesis = split_pair(input_text)

    # top-k by absolute contribution
    topk = [w for (w, s) in sorted(lime_attributions, key=lambda x: abs(x[1]), reverse=True)[:k]]

    def remove_tokens_regex(text: str, tokens: list[str]) -> str:
        out = text
        for t in sorted(set(tokens), key=len, reverse=True):
            out = re.sub(rf"\b{re.escape(t)}\b", " ", out, flags=re.IGNORECASE)
        return re.sub(r"\s{2,}", " ", out).strip()

    def keep_only_tokens_regex(text: str, tokens: list[str]) -> str:
        keep = {t.lower() for t in tokens}
        toks = re.findall(r"\w+|\S", text)
        kept = [tok for tok in toks if tok.lower() in keep]
        if not kept:
            return ""
        return re.sub(r"\s{2,}", " ", " ".join(kept)).strip()

    hyp_drop = remove_tokens_regex(hypothesis, topk)
    hyp_keep = keep_only_tokens_regex(hypothesis, topk)

    p_orig = clf.predict_proba([join_pair(premise, hypothesis)])[0]
    yhat = int(p_orig.argmax())

    p_drop = clf.predict_proba([join_pair(premise, hyp_drop)])[0]
    p_keep = clf.predict_proba([join_pair(premise, hyp_keep)])[0]

    return {
        "faithfulness": float(p_orig[yhat] - p_drop[yhat]),
        "comprehensiveness": float(p_orig[yhat] - p_drop[yhat]),
        "sufficiency": float(p_keep[yhat]),
        "topk": topk
    }

def remove_tokens_regex(text: str, tokens: list[str]) -> str:
    out = text
    for t in sorted(set(tokens), key=len, reverse=True):
        out = re.sub(rf"\b{re.escape(t)}\b", " ", out, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", out).strip()

def keep_only_tokens_regex(text: str, tokens: list[str]) -> str:
    keep = {t.lower() for t in tokens}
    toks = re.findall(r"\w+|\S", text)
    kept = [tok for tok in toks if tok.lower() in keep]
    if not kept:
        return ""
    return re.sub(r"\s{2,}", " ", " ".join(kept)).strip()

def compute_faithfulness(classifier, original_text, top_tokens):
    try:
        p, h = split_pair(original_text)
        hyp_drop = remove_tokens_regex(h, top_tokens)
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_drop = classifier.predict_proba([join_pair(p, hyp_drop)])[0]
        return float(p_orig[yhat] - p_drop[yhat])   # larger = tokens mattered
    except Exception:
        return 0.0

def compute_sufficiency(classifier, original_text, top_tokens):
    try:
        p, h = split_pair(original_text)
        kept_h = keep_only_tokens_regex(h, top_tokens)
        if not kept_h.strip():
            return 0.0
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_keep = classifier.predict_proba([join_pair(p, kept_h)])[0]
        return float(p_keep[yhat])   # larger = kept tokens suffice
    except Exception:
        return 0.0

def compute_comprehensiveness(classifier, original_text, top_tokens):
    try:
        p, h = split_pair(original_text)
        rem_h = remove_tokens_regex(h, top_tokens)
        p_orig = classifier.predict_proba([original_text])[0]
        yhat = int(np.argmax(p_orig))
        p_rem = classifier.predict_proba([join_pair(p, rem_h)])[0]
        return float(p_orig[yhat] - p_rem[yhat])   # same delta form as faithfulness
    except Exception:
        return 0.0

In [29]:
def compute_evaluation_metrics(classifier, lime_results, k=3, dirs=None, save_every=10):
    """Compute faithfulness, sufficiency, and comprehensiveness"""
    print(f"📊 Computing evaluation metrics (k={k})...")
    out_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "metrics_partial.json") if dirs else None

    done_ids = set()
    partial = []
    if out_path and file_nonempty(out_path):
        try:
            raw = read_json(out_path)
            partial = raw.get("data", raw) if isinstance(raw, dict) else raw
            done_ids = {int(r["id"]) for r in partial if "id" in r}
            print(f"⏩ Resuming metrics from {len(done_ids)} items.")
        except Exception:
            pass

    results = list(partial)
    since_last = 0
    
    for result in tqdm(lime_results, desc="Computing metrics"):
        try:
            text = result["input_text"]
            attributions = result.get("lime_attributions_filtered", result["lime_attributions"])
            
            if len(attributions) < k:
                continue

            rid = int(result["id"])
            if rid in done_ids:
                continue

            # choose attrs safely
            attrs = _select_attributions(result)
            top_tokens = [w for (w, _) in attrs[:k]]

            # split and mutate hypothesis only
            premise, hypothesis = split_pair(result["input_text"])

            # effective_k (for logging/debug)
            effective_k = sum(bool(re.search(rf"\b{re.escape(t)}\b", hypothesis, re.IGNORECASE)) for t in top_tokens)

            # build mutants
            hyp_drop = remove_tokens_regex(hypothesis, top_tokens) if top_tokens else hypothesis
            hyp_keep = keep_only_tokens_regex(hypothesis, top_tokens) if top_tokens else ""

            # predictions
            p_orig = classifier.predict_proba([join_pair(premise, hypothesis)])[0]
            yhat   = int(np.argmax(p_orig))
            p_drop = classifier.predict_proba([join_pair(premise, hyp_drop)])[0] if hyp_drop else p_orig
            p_keep = (classifier.predict_proba([join_pair(premise, hyp_keep)])[0]
                    if hyp_keep != "" else np.zeros_like(p_orig))

            # raw metrics
            faith = float(p_orig[yhat] - p_drop[yhat])         # larger = tokens mattered
            comp  = float(p_orig[yhat] - p_drop[yhat])         # same delta form (your usage)
            suff  = float(p_keep[yhat])                        # larger = kept tokens suffice

            # sanitize to numbers (no NaNs/None)
            for name, val in (("faithfulness", faith), ("comprehensiveness", comp), ("sufficiency", suff)):
                if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
                    locals()[name[:4]] = 0.0  # set to 0.0 on weirdness

            metrics_row = {
                "faithfulness": faith,
                "sufficiency":  suff,
                "comprehensiveness": comp,
                "effective_k": int(effective_k),
                "used_filtered_attributions": bool(result.get("lime_attributions_filtered")),
                "top_tokens": top_tokens,
            }

            results.append({
                "id": rid,
                **metrics_row
            })
            done_ids.add(rid)
            since_last += 1
            if out_path and since_last >= save_every:
                write_json(out_path, with_schema({"data": results}))
                print(f"💾 Checkpoint: saved {len(results)} metrics items to {out_path}")
                since_last = 0
            
        except Exception as e:
            metrics_row = {
                "faithfulness": 0.0,
                "sufficiency":  0.0,
                "comprehensiveness": 0.0,
                "effective_k": 0,
                "used_filtered_attributions": bool(result.get("lime_attributions_filtered")),
                "top_tokens": [],
                "error": f"{type(e).__name__}: {e}",
            }
            print(f"❌ Error computing metrics for example {result['id']}: {e}")
            continue
    
    write_json(out_path, with_schema({"data": results}))
    
    return results

In [30]:
def perform_sanity_checks(classifier, lime_results, dirs):
    """Perform sanity checks on explanations"""
    print("🔍 Performing sanity checks...")
    
    src_hash = compute_hash([r["id"] for r in lime_results])
    sanity_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "sanity_check_results.json")
    reuse, prev = should_reuse(sanity_path, {"source": "lime_filtered", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing sanity checks from: {sanity_path}")
        return prev
    
    # Separate correct and incorrect predictions
    to_mnli = {'contradiction':0,'neutral':1,'entailment':2}
    correct_examples = [r for r in lime_results if to_mnli[r['true_label_name']] == int(r['predicted_label'])]
    incorrect_examples = [r for r in lime_results if to_mnli[r['true_label_name']] != int(r['predicted_label'])]
    
    print(f"Found {len(correct_examples)} correct, {len(incorrect_examples)} incorrect predictions")
    
    sanity_results = {
        "correct_examples": [],
        "incorrect_examples": [],
        "shuffle_test_results": []
    }
    
    # Check top 5 correct examples
    for i, example in enumerate(correct_examples[:5]):
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_5_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:5]
        
        sanity_results["correct_examples"].append({
            "id": example["id"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_tokens": [(token, float(score)) for token, score in top_5_tokens],
            "premise": example["premise"][:100] + "..." if len(example["premise"]) > 100 else example["premise"],
            "hypothesis": example["hypothesis"][:100] + "..." if len(example["hypothesis"]) > 100 else example["hypothesis"]
        })
    
    # Check top 5 incorrect examples
    for i, example in enumerate(incorrect_examples[:5]):
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_5_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:5]
        
        sanity_results["incorrect_examples"].append({
            "id": example["id"],
            "true_class": example["true_label_name"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_tokens": [(token, float(score)) for token, score in top_5_tokens],
            "premise": example["premise"][:100] + "..." if len(example["premise"]) > 100 else example["premise"],
            "hypothesis": example["hypothesis"][:100] + "..." if len(example["hypothesis"]) > 100 else example["hypothesis"]
        })
    
    # Shuffle test on 3 examples
    print("🔀 Running shuffle sanity test...")
    for i, example in enumerate(lime_results[:3]):
        original_text = example["input_text"]
        
        # Shuffle *hypothesis only*
        p, h = split_pair(original_text)
        words = h.split()
        random.shuffle(words)
        shuffled_text = join_pair(p, " ".join(words))
        
        # Compute faithfulness for both
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        if len(attributions) >= 3:
            top_tokens = [token for token, _ in sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:3]]
            
            original_faithfulness = compute_faithfulness(classifier, original_text, top_tokens)
            shuffled_faithfulness = compute_faithfulness(classifier, shuffled_text, top_tokens)
            
            sanity_results["shuffle_test_results"].append({
                "id": example["id"],
                "original_faithfulness": float(original_faithfulness),
                "shuffled_faithfulness": float(shuffled_faithfulness),
                "faithfulness_drop": float(original_faithfulness - shuffled_faithfulness)
            })
    
    # Save sanity check results
    write_json(sanity_path, {"schema_version": SCHEMA_VERSION,
                             "meta": {"source": "lime_filtered", "source_hash": src_hash},
                             **sanity_results})
    print(f"✅ Sanity check results saved to: {sanity_path}")
    
    return sanity_results

In [31]:
def error_analysis(lime_results, dirs):
    """Analyze errors and create confusion matrix"""
    print("🔍 Performing error analysis...")
    
    src_hash = compute_hash([ (r["id"], r["true_label_name"], r["predicted_class"]) for r in lime_results ])
    error_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "error_analysis.json")
    reuse, prev = should_reuse(error_path, {"source": "lime_filtered", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing error analysis from: {error_path}")
        return prev
    
    # Get 10 most confident wrong predictions
    wrong_predictions = [r for r in lime_results if r['true_label_name'] != r['predicted_class']]
    most_confident_wrong = sorted(wrong_predictions, key=lambda x: x['confidence'], reverse=True)[:10]
    
    error_analysis_results = []
    for example in most_confident_wrong:
        attributions = example.get("lime_attributions_filtered", example["lime_attributions"])
        top_10_tokens = sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:10]
        
        error_analysis_results.append({
            "id": example["id"],
            "premise": example["premise"],
            "hypothesis": example["hypothesis"],
            "true_label_name": example["true_label_name"],
            "predicted_class": example["predicted_class"],
            "confidence": example["confidence"],
            "top_10_attributions": [(token, float(score)) for token, score in top_10_tokens]
        })
    
    # Create confusion matrix
    y_true = [SNLI_TO_MNLI_IDX[int(r['true_label'])] for r in lime_results]   # remap SNLI -> MNLI index space
    y_pred = [int(r['predicted_label']) for r in lime_results]
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])  # MNLI indices: 0=contradiction, 1=neutral, 2=entailment
    label_names = ['entailment', 'neutral', 'contradiction']
    
    # Save error analysis
    error_data = {
        "schema_version": SCHEMA_VERSION,
        "meta": {
            "source": "lime_filtered", 
            "source_hash": src_hash
        },
        "most_confident_wrong_predictions": error_analysis_results,
        "confusion_matrix": cm.tolist(),
        "label_names": label_names,
        "summary": {
            "total_examples": len(lime_results),
            "wrong_predictions": len(wrong_predictions),
            "error_rate": len(wrong_predictions) / len(lime_results)
        }
    }
    
    write_json(error_path, error_data)
    print(f"✅ Error analysis saved to: {error_path}")
    
    return error_data

In [32]:
def lime_stability_sweep(classifier, processed_snli, dirs):
    """Mini parameter sweep for LIME stability"""
    print("🔬 Running LIME stability mini-sweep...")
    
    stability_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_stability_sweep.json")
    # Parameter grid
    param_grid = [
        {"num_samples": 500, "kernel_width": None},
        {"num_samples": 1000, "kernel_width": None},
        # {"num_samples": 500, "kernel_width": 25},
        # {"num_samples": 1000, "kernel_width": 25}
    ]
    
    # Test on 10 examples
    test_examples = processed_snli[:10]  # keep small so it’s fast/visible

    src_hash = compute_hash({"texts":[ex["input_text"] for ex in test_examples], "grid": param_grid})
    reuse, prev = should_reuse(stability_path, {"source": "processed_snli_subset", "source_hash": src_hash})
    if reuse:
        print(f"⏩ Reusing stability sweep from: {stability_path}")
        return prev.get("results", [])
    
    total_tasks = len(param_grid) * len(test_examples)

    stability_results = []
    with tqdm(total=total_tasks, desc="Stability sweep", unit="ex", dynamic_ncols=True) as pbar:
        for pi, params in enumerate(param_grid, 1):
            kw = params["kernel_width"]
            kw_label = "default" if kw is None else kw

            # Build explainer (omit kernel_width when None)
            init_kwargs = {
                "class_names": list(classifier.label_mapping.values()),
                "verbose": False,
                "random_state": 42,
            }
            if kw is not None:
                init_kwargs["kernel_width"] = float(kw)

            metrics = []
            for ei, ex in enumerate(test_examples, 1):
                try:
                    text = ex["input_text"]
                    text = ex["input_text"]
                    p, h = split_pair(text)
                    attributions, _ = explain_hypothesis_lime(
                        premise=p,
                        hypothesis=h,
                        clf_predict_proba=classifier.predict_proba,
                        num_features=10,
                        num_samples=int(params["num_samples"]),
                        split_expression=r"\s+"
                    )

                    if len(attributions) >= 3:
                        top_tokens = [t for t, _ in sorted(attributions, key=lambda x: abs(x[1]), reverse=True)[:3]]
                        f = compute_faithfulness(classifier, text, top_tokens)
                        s = compute_sufficiency(classifier, text, top_tokens)
                        metrics.append({"faithfulness": float(f), "sufficiency": float(s)})

                except Exception as e:
                    tqdm.write(f"Error (ns={params['num_samples']}, kw={kw_label}, ex={ei}): {e}")

                # update the global bar + postfix info
                pbar.set_postfix_str(f"ns={params['num_samples']}, kw={kw_label}, ex {ei}/{len(test_examples)}")
                pbar.update(1)

            if metrics:
                stability_results.append({
                    "params": params,
                    "avg_faithfulness": float(np.mean([m["faithfulness"] for m in metrics])),
                    "avg_sufficiency": float(np.mean([m["sufficiency"] for m in metrics])),
                    "num_examples": len(metrics),
                })
                write_json(stability_path, with_schema({"results": stability_results}))
    
    # Save stability results
    stability_data = {
        "schema_version": SCHEMA_VERSION,
        "meta": {
            "source":"processed_snli_subset",
            "source_hash": src_hash
        },
        "results": stability_results
    }
    write_json(stability_path, stability_data)
    print(f"✅ Stability sweep results saved to: {stability_path}")
    
    return stability_results

In [33]:
def bootstrap_confidence_intervals(metrics_results, n_bootstrap=1000):
    """Compute bootstrap confidence intervals for metrics"""
    if not metrics_results:
        return {}
    
    print("🔄 Computing bootstrap confidence intervals...")
    
    metrics_df = pd.DataFrame(metrics_results)
    bootstrap_results = {}
    
    for metric in ['faithfulness', 'sufficiency', 'comprehensiveness']:
        if metric in metrics_df.columns:
            values = metrics_df[metric].values
            bootstrap_means = []
            
            for _ in range(n_bootstrap):
                # Resample with replacement
                resampled = np.random.choice(values, size=len(values), replace=True)
                bootstrap_means.append(np.mean(resampled))
            
            # Calculate 95% CI
            ci_lower = np.percentile(bootstrap_means, 2.5)
            ci_upper = np.percentile(bootstrap_means, 97.5)
            
            bootstrap_results[metric] = {
                "mean": float(np.mean(values)),
                "ci_lower": float(ci_lower),
                "ci_upper": float(ci_upper),
                "std": float(np.std(bootstrap_means))
            }
    
    print("✅ Bootstrap confidence intervals computed")
    return bootstrap_results

def generate_report(lime_results, metrics_results, dirs, config,
                    sanity_results=None, error_results=None, stability_results=None):
    """Generate comprehensive markdown report (idempotent + uses sanity/error/stability)"""
    import os, json, math
    import numpy as np
    import pandas as pd

    print("📝 Generating SNLI LIME report...")

    # fall back to disk if None (keeps resumability)
    if sanity_results is None:
        path = os.path.join(dirs['LIME_OUTPUT_DIR'], 'sanity_check_results.json')
        sanity_results = read_json(path) if file_nonempty(path) else {}
    if error_results is None:
        path = os.path.join(dirs['LIME_OUTPUT_DIR'], 'error_analysis.json')
        error_results = read_json(path) if file_nonempty(path) else {}
    if stability_results is None:
        path = os.path.join(dirs['LIME_OUTPUT_DIR'], 'lime_stability_sweep.json')
        stability_raw = read_json(path) if file_nonempty(path) else {}
        # support either {"results":[...]} or a flat list
        stability_results = stability_raw.get("results", stability_raw if isinstance(stability_raw, list) else [])

    report_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "SNLI_LIME_report.md")
    src_hash = compute_hash([r["id"] for r in lime_results] + [m["id"] for m in (metrics_results or [])])
    stamp_path = report_path + ".meta.json"
    reuse, _ = should_reuse(stamp_path, {"source": "lime+metrics", "source_hash": src_hash})
    if reuse and file_nonempty(report_path):
        print(f"⏩ Report up-to-date; skipping.")
        with open(report_path, "r", encoding="utf-8") as f:
            return f.read()

    # Safe config access with defaults
    def cfg(key, default="NA"):
        return config.get(key, default) if isinstance(config, dict) else default

    # Basic stats
    total_examples = len(lime_results) or 1
    correct_predictions = sum(1 for r in lime_results if r.get('true_label_name') == r.get('predicted_class'))
    accuracy = correct_predictions / total_examples
    avg_confidence = float(np.mean([float(r.get('confidence', 0.0)) for r in lime_results])) if lime_results else float('nan')

    # Metrics statistics
    metrics_stats = {}
    if metrics_results:
        metrics_df = pd.DataFrame(metrics_results)
        for metric in ['faithfulness', 'sufficiency', 'comprehensiveness']:
            if metric in metrics_df.columns:
                col = pd.to_numeric(metrics_df[metric], errors='coerce')
                metrics_stats[metric] = {
                    'mean': float(col.mean()),
                    'std': float(col.std()),
                    'min': float(col.min()),
                    'max': float(col.max())
                }

    # Bootstrap CIs (assumes helper exists; otherwise skip gracefully)
    try:
        bootstrap_cis = bootstrap_confidence_intervals(metrics_results)
    except Exception:
        bootstrap_cis = {}

    # Qualitative examples
    correct_examples = [r for r in lime_results if r.get('true_label_name') == r.get('predicted_class')]
    incorrect_examples = [r for r in lime_results if r.get('true_label_name') != r.get('predicted_class')]
    top_correct = sorted(correct_examples, key=lambda x: x.get('confidence', 0.0), reverse=True)[:3]
    top_incorrect = sorted(incorrect_examples, key=lambda x: x.get('confidence', 0.0), reverse=True)[:2]

    # Sanity summaries
    sane_correct = sanity_results.get("correct", correct_predictions)
    sane_incorrect = sanity_results.get("incorrect", total_examples - correct_predictions)

    # Error analysis summaries
    top_patterns = []
    if isinstance(error_results, dict):
        # support {"top_patterns": {"pattern": count, ...}}
        tp = error_results.get("top_patterns", {})
        if isinstance(tp, dict):
            top_patterns = sorted(tp.items(), key=lambda x: x[1], reverse=True)[:10]

    # Stability summaries
    sweep_summary_lines = []
    if isinstance(stability_results, list) and stability_results:
        # group by params
        from collections import defaultdict
        by_cfg = defaultdict(list)
        for r in stability_results:
            p = r.get("params", {})
            key = (int(p.get("num_samples", -1)), p.get("kernel_width", None))
            by_cfg[key].append(float(r.get("lime_score", 0.0)))
        for (ns, kw), scores in sorted(by_cfg.items()):
            if scores:
                sweep_summary_lines.append(
                    f"- num_samples={ns}, kernel_width={kw}: "
                    f"avg_fidelity={np.mean(scores):.3f} (n={len(scores)})"
                )

    # Build Markdown
    report_content = []
    report_content += [f"# SNLI LIME Explanation Analysis Report\n"]
    report_content += [f"**Dataset**: SNLI  \n"
                       f"**Model**: {cfg('model_name')}  \n"
                       f"**Analysis Date**: {cfg('timestamp','')[:10]}  \n"
                       f"**Total Examples**: {total_examples}  \n"
                       f"**Git Hash**: {cfg('git_hash')}\n"]

    report_content += [f"## Key Results\n",
                       f"- **Model Accuracy**: {accuracy:.3f} ({correct_predictions}/{total_examples})\n",
                       f"- **Average Confidence**: {avg_confidence:.3f}\n",
                       f"- **LIME Parameters**: {cfg('lime_num_samples')} samples, {cfg('lime_num_features')} features\n",
                       f"- **Random Seed**: {cfg('random_seed')}\n"]

    if metrics_stats:
        report_content += ["\n## Attribution Metrics Results\n",
                           "| Metric | Mean | Std | Min | Max | 95% CI |\n",
                           "|--------|-----:|----:|----:|----:|:------:|\n"]
        for metric, stats in metrics_stats.items():
            ci_info = ""
            if metric in bootstrap_cis:
                ci = bootstrap_cis[metric]
                ci_info = f"[{ci.get('ci_lower', float('nan')):.3f}, {ci.get('ci_upper', float('nan')):.3f}]"
            report_content += [f"| {metric.title()} | {stats['mean']:.3f} | {stats['std']:.3f} | "
                               f"{stats['min']:.3f} | {stats['max']:.3f} | {ci_info} |\n"]

    report_content += ["\n## Sanity Checks\n",
                       f"- Correct predictions: **{sane_correct}**\n",
                       f"- Incorrect predictions: **{sane_incorrect}**\n"]

    report_content += ["\n## Error Analysis (Top Patterns)\n"]
    if top_patterns:
        for pat, cnt in top_patterns:
            report_content += [f"- `{pat}` — {cnt}\n"]
    else:
        report_content += ["- No pattern summary available.\n"]

    report_content += ["\n## LIME Stability Sweep (mini)\n"]
    if sweep_summary_lines:
        report_content += [*sweep_summary_lines, "\n"]
    else:
        report_content += ["- No stability results available.\n"]

    report_content += ["\n## LIME Configuration\n",
                       f"- **Number of Samples**: {cfg('lime_num_samples')}\n",
                       f"- **Number of Features**: {cfg('lime_num_features')}\n",
                       f"- **Top-k Tokens for Metrics**: {cfg('k_tokens')}\n",
                       f"- **Chunk Size**: {cfg('chunk_size')}\n"]

    report_content += ["\n## Qualitative Examples — Top Correct Predictions\n"]
    for i, example in enumerate(top_correct, start=1):
        atts = example.get('lime_attributions_filtered', example.get('lime_attributions', []))
        top_tokens = sorted(atts, key=lambda x: abs(x[1]), reverse=True)[:5]
        token_strs = ", ".join([f"{t}({s:+.3f})" for t, s in top_tokens])
        report_content += [
            f"**Example {i}** (Conf: {example.get('confidence', 0.0):.3f})\n",
            f"- **Premise**: {example.get('premise','')}\n",
            f"- **Hypothesis**: {example.get('hypothesis','')}\n",
            f"- **Predicted**: {example.get('predicted_class','')}\n",
            f"- **Top Attributions**: {token_strs}\n\n"
        ]

    if top_incorrect:
        report_content += ["\n## Qualitative Examples — Top Incorrect Predictions\n"]
        for i, example in enumerate(top_incorrect, start=1):
            atts = example.get('lime_attributions_filtered', example.get('lime_attributions', []))
            top_tokens = sorted(atts, key=lambda x: abs(x[1]), reverse=True)[:5]
            token_strs = ", ".join([f"{t}({s:+.3f})" for t, s in top_tokens])
            report_content += [
                f"**Example {i}** (Conf: {example.get('confidence', 0.0):.3f})\n",
                f"- **Premise**: {example.get('premise','')}\n",
                f"- **Hypothesis**: {example.get('hypothesis','')}\n",
                f"- **True**: {example.get('true_label_name','')}  \n",
                f"- **Predicted**: {example.get('predicted_class','')}\n",
                f"- **Top Attributions**: {token_strs}\n\n"
            ]

    report_content += ["\n## Known Caveats\n",
                       f"- **Delimiter Handling**: Uses `{SEP_WITH_SPACES}` separator between premise and hypothesis\n",
                       "- **Stopword Filtering**: Removes common English stopwords and special tokens\n",
                       f"- **Sample Size**: Analysis based on **{total_examples}** examples\n",
                       "- **Perturbation Sensitivity**: LIME results may vary with different num_samples settings\n"]

    report_content += ["\n## Next Steps\n",
                       "1. Add `kernel_width` variant to stability on a small subset (e.g., 50 ex)\n",
                       "2. Calibrate confidence (reliability diagram) and note over-/under-confidence\n",
                       "3. Compare with SHAP/Integrated Gradients on the same subset\n"]

    # Save real Markdown
    content = "".join(report_content)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(content)

    # stamp for idempotence
    write_json(stamp_path, {"schema_version": SCHEMA_VERSION,
                            "meta": {"source": "lime+metrics", "source_hash": src_hash}})
    print(f"✅ Report saved to: {report_path}")
    return content

def analyze_results_enhanced(lime_results, metrics_results, dirs):
    """Enhanced results analysis with comprehensive statistics"""
    print("📈 Enhanced results analysis...")
    
    if not lime_results:
        print("⏩ No results; skipping analysis.")
        return {}
    
    # Basic performance metrics
    total = len(lime_results)
    correct = sum(1 for r in lime_results if r.get("true_label_name") == r.get("predicted_class"))
    accuracy = float(correct / total) if total else 0.0
    avg_confidence = float(np.mean([r.get("confidence", 0) for r in lime_results]))
    
    # Metrics analysis
    metrics_summary = {}
    if metrics_results:
        metrics_df = pd.DataFrame(metrics_results)
        metrics_summary = {
            "avg_faithfulness": float(metrics_df['faithfulness'].mean()),
            "avg_sufficiency": float(metrics_df['sufficiency'].mean()),
            "avg_comprehensiveness": float(metrics_df['comprehensiveness'].mean()),
            "std_faithfulness": float(metrics_df['faithfulness'].std()),
            "std_sufficiency": float(metrics_df['sufficiency'].std()),
            "std_comprehensiveness": float(metrics_df['comprehensiveness'].std()),
            "num_examples": len(metrics_results)
        }
    
    # Confidence distribution analysis
    confidences = [r['confidence'] for r in lime_results]
    confidence_analysis = {
        "high_confidence": sum(1 for c in confidences if c > 0.8),
        "medium_confidence": sum(1 for c in confidences if 0.5 < c <= 0.8),
        "low_confidence": sum(1 for c in confidences if c <= 0.5),
        "avg_confidence": float(np.mean(confidences)),
        "std_confidence": float(np.std(confidences))
    }
    
    # Build final results
    final_results = {
        "schema_version": SCHEMA_VERSION,
        "evaluation_metrics": metrics_summary,
        "model_performance": {
            "accuracy": accuracy,
            "avg_confidence": avg_confidence,
            "correct_predictions": int(correct),
            "total_examples": int(total),
            "error_rate": float(1 - accuracy)
        },
        "confidence_analysis": confidence_analysis,
        "lime_explanations": lime_results,
        "individual_metrics": metrics_results if metrics_results else []
    }

    # Attach calibration if available (written by main)
    cal_path = os.path.join(dirs["LIME_OUTPUT_DIR"], "calibration.json")
    if file_nonempty(cal_path):
        try:
            final_results["calibration"] = read_json(cal_path)
        except Exception:
            pass
    
    # Save results
    results_path = os.path.join(dirs["LIME_OUTPUT_DIR"], "complete_evaluation_results.json")
    write_json(results_path, final_results)
    
    print(f"\n🎯 FINAL EVALUATION RESULTS:")
    print(f"   Model Accuracy: {accuracy:.3f}")
    print(f"   Average Confidence: {avg_confidence:.3f}")
    
    if metrics_summary:
        print(f"   Average Faithfulness: {metrics_summary['avg_faithfulness']:.3f} ± {metrics_summary['std_faithfulness']:.3f}")
        print(f"   Average Sufficiency: {metrics_summary['avg_sufficiency']:.3f} ± {metrics_summary['std_sufficiency']:.3f}")
        print(f"   Average Comprehensiveness: {metrics_summary['avg_comprehensiveness']:.3f} ± {metrics_summary['std_comprehensiveness']:.3f}")
    
    print(f"✅ Complete results saved to: {results_path}")
    return final_results

def plot_runtime_distribution(lime_results, dirs):
    print("📈 Plotting LIME runtime distribution...")
    times = [r.get("lime_runtime_sec", np.nan) for r in lime_results]
    times = [t for t in times if not np.isnan(t)]
    if not times: return
    plt.figure(figsize=(8,5))
    plt.hist(times, bins=20, edgecolor="black", alpha=0.7)
    plt.title("Per-example LIME runtime")
    plt.xlabel("Seconds"); plt.ylabel("Count")
    outp = os.path.join(dirs['LIME_OUTPUT_DIR'], "plots", "lime_runtime_hist.png")
    ensure_dir(os.path.dirname(outp))
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def plot_per_class_top_words(lime_results, dirs, by="predicted_class", top_n=15):
    print("📈 Plotting top words per class...")
    groups = {"entailment": [], "neutral": [], "contradiction": []}
    for r in lime_results:
        lbl = r.get(by)
        if lbl not in groups: continue
        atts = r.get("lime_attributions_filtered", r.get("lime_attributions", []))
        groups[lbl].extend([ (w.lower(), abs(s)) for w,s in atts ])

    for lbl, pairs in groups.items():
        if not pairs: continue
        from collections import defaultdict
        agg = defaultdict(list)
        for w,s in pairs: agg[w].append(s)
        avg = sorted(((w, float(np.mean(v))) for w,v in agg.items()),
                     key=lambda x: x[1], reverse=True)[:top_n]
        words, scores = zip(*avg)
        plt.figure(figsize=(8,6))
        plt.barh(range(len(words)), scores)
        plt.yticks(range(len(words)), words)
        plt.gca().invert_yaxis()
        plt.title(f"Top {top_n} contributing words — {lbl}")
        plt.xlabel("Avg |attribution|")
        outp = os.path.join(dirs['LIME_OUTPUT_DIR'], "plots", f"top_words_{lbl}.png")
        ensure_dir(os.path.dirname(outp))
        plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def plot_reliability_diagram(lime_results, dirs, bins=10):
    print("📈 Plotting reliability diagram...")
    # max prob & correctness
    # Normalize lime_results if wrapped
    if isinstance(lime_results, dict) and "data" in lime_results:
        lime_results = lime_results["data"]

    preds, correct = [], []
    for r in lime_results:
        ap = r.get("all_probabilities")
        if isinstance(ap, dict):
            conf = ap.get(r.get("predicted_class"), r.get("confidence", 0.0))
        else:
            conf = r.get("confidence", 0.0)
        preds.append(float(conf))
        correct.append(1.0 if r.get("predicted_class") == r.get("true_label_name") else 0.0)

    preds = np.array(preds, dtype=float)
    correct = np.array(correct, dtype=float)

    # bin by confidence
    edges = np.linspace(0, 1, bins+1)
    idx = np.digitize(preds, edges) - 1
    acc, conf, cnt = [], [], []
    for b in range(bins):
        mask = idx == b
        if mask.any():
            acc.append(float(correct[mask].mean()))
            conf.append(float(preds[mask].mean()))
            cnt.append(int(mask.sum()))
        else:
            acc.append(np.nan); conf.append((edges[b]+edges[b+1])/2); cnt.append(0)

    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1], '--', label='perfect')
    plt.plot(conf, acc, marker='o', label='model')
    plt.xlabel('Mean predicted confidence'); plt.ylabel('Empirical accuracy')
    plt.title('Reliability diagram'); plt.legend()
    outp = os.path.join(dirs['LIME_OUTPUT_DIR'], "plots", "reliability.png")
    ensure_dir(os.path.dirname(outp))
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()

def compute_and_save_calibration(classifier, processed_records, dirs, n_bins=15, val_fraction=0.2):
    """
    Computes ECE before/after temperature scaling on a held-out tail split.
    Writes {LIME_OUTPUT_DIR}/calibration.json and returns the dict.
    """
    import numpy as np, torch, torch.nn as nn, os

    def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
        conf = probs.max(axis=1); preds = probs.argmax(axis=1); correct = (preds == labels).astype(float)
        edges = np.linspace(0.0, 1.0, n_bins + 1); ece = 0.0
        for i in range(n_bins):
            in_bin = (conf > edges[i]) & (conf <= edges[i+1])
            if not np.any(in_bin): continue
            ece += in_bin.mean() * abs(correct[in_bin].mean() - conf[in_bin].mean())
        return float(ece)

    class TempScaler(nn.Module):
        def __init__(self): super().__init__(); self.log_T = nn.Parameter(torch.zeros(1))
        def forward(self, logits): return logits / torch.exp(self.log_T)

    def fit_temperature(logits_t: torch.Tensor, labels_t: torch.Tensor, max_iter=100) -> float:
        scaler = TempScaler().to(logits_t.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS(scaler.parameters(), lr=0.5, max_iter=max_iter)
        def closure():
            optimizer.zero_grad()
            loss = criterion(scaler(logits_t), labels_t); loss.backward(); return loss
        optimizer.step(closure)
        return float(torch.exp(scaler.log_T).item())

    def apply_temperature_np(logits_np: np.ndarray, T: float) -> np.ndarray:
        logits_t = torch.from_numpy(logits_np).float() / T
        return torch.softmax(logits_t, dim=-1).numpy()

    N = len(processed_records)
    if N == 0:
        return {}

    val_size = max(1, int(N * val_fraction))
    texts_val = [join_pair(r["premise"], r["hypothesis"]) for r in processed_records[-val_size:]]
    labels_val = np.array([SNLI_TO_MNLI_IDX[r["label"]] for r in processed_records[-val_size:]], dtype=int)

    logits_val = classifier.predict_logits(texts_val)
    probs_val = torch.softmax(torch.from_numpy(logits_val), dim=-1).numpy()

    ece_raw = expected_calibration_error(probs_val, labels_val, n_bins=n_bins)
    T = fit_temperature(torch.from_numpy(logits_val).float(), torch.from_numpy(labels_val).long())
    probs_temp = apply_temperature_np(logits_val, T)
    ece_temp = expected_calibration_error(probs_temp, labels_val, n_bins=n_bins)

    out = {"ece_raw": float(ece_raw), "ece_temp": float(ece_temp), "temperature": float(T), "val_size": int(val_size)}
    cal_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "calibration.json")
    write_json(cal_path, out)
    return out

def plot_reliability_diagram_with_temp(lime_results, dirs, classifier, temperature, bins=10):
    """Overlay temp-scaled curve on the reliability diagram."""
    import numpy as np

    # Raw from stored probabilities
    preds_raw = np.array([r["all_probabilities"].get(r["predicted_class"], r["confidence"])
                          for r in lime_results], dtype=float)
    correct_raw = np.array([int(r["true_label_name"] == r["predicted_class"]) for r in lime_results], dtype=float)

    # Temp-scaled from logits
    texts = [r["input_text"] for r in lime_results]
    logits = classifier.predict_logits(texts)
    logits_t = torch.from_numpy(logits).float() / float(temperature)
    probs_t = torch.softmax(logits_t, dim=-1).numpy()
    y_true = np.array([MNLI_LABELS.index(r["true_label_name"]) for r in lime_results], dtype=int)
    conf_t = probs_t.max(axis=1); corr_t = (probs_t.argmax(axis=1) == y_true).astype(float)

    edges = np.linspace(0, 1, bins+1)

    def binned(conf, corr):
        idx = np.digitize(conf, edges) - 1
        acc, cbar = [], []
        for b in range(bins):
            mask = idx == b
            if mask.any():
                acc.append(float(corr[mask].mean())); cbar.append(float(conf[mask].mean()))
            else:
                acc.append(np.nan); cbar.append((edges[b]+edges[b+1])/2)
        return cbar, acc

    conf_raw, acc_raw = binned(preds_raw, correct_raw)
    conf_tmp, acc_tmp = binned(conf_t, corr_t)

    plt.figure(figsize=(6,6))
    plt.plot([0,1],[0,1], '--', label='perfect')
    plt.plot(conf_raw, acc_raw, marker='o', label='raw')
    plt.plot(conf_tmp, acc_tmp, marker='s', label='temp-scaled')
    plt.xlabel('Mean predicted confidence'); plt.ylabel('Empirical accuracy')
    plt.title('Reliability diagram (raw vs temp-scaled)'); plt.legend()
    outp = os.path.join(dirs['LIME_OUTPUT_DIR'], "plots", "reliability_with_temp.png")
    ensure_dir(os.path.dirname(outp))
    plt.tight_layout(); plt.savefig(outp, dpi=300); plt.close()


In [34]:
def create_comprehensive_plots(lime_results, metrics_results, dirs, classifier=None):
    """Create comprehensive visualization plots"""
    print("📊 Creating comprehensive visualization plots...")

    src_hash = compute_hash([ (r["id"], r.get("lime_attributions_filtered", [])) for r in lime_results ])
    plots_dir = os.path.join(dirs['LIME_OUTPUT_DIR'], "plots")
    ensure_dir(plots_dir)
    stamp = os.path.join(plots_dir, ".plots.meta.json")
    reuse, _ = should_reuse(stamp, {"source":"lime_filtered","source_hash": src_hash})
    if reuse:
        print(f"⏩ Plots up-to-date; skipping.")
        return
    
    # Set style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # 1. Model Performance Analysis
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Confidence distribution
    confidences = [r['confidence'] for r in lime_results]
    axes[0, 0].hist(confidences, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Model Confidence Distribution')
    axes[0, 0].set_xlabel('Confidence Score')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].axvline(np.mean(confidences), color='red', linestyle='--', 
                       label=f'Mean: {np.mean(confidences):.3f}')
    axes[0, 0].legend()
    
    # Label distribution
    true_labels = [r['true_label_name'] for r in lime_results]
    pred_labels = [r['predicted_class'] for r in lime_results]
    
    label_counts = Counter(true_labels)
    axes[0, 1].bar(label_counts.keys(), label_counts.values(), alpha=0.7, color='lightgreen')
    axes[0, 1].set_title('True Label Distribution')
    axes[0, 1].set_xlabel('Labels')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Accuracy by confidence bins
    df_results = pd.DataFrame(lime_results)
    df_results['correct'] = df_results['true_label_name'] == df_results['predicted_class']
    df_results['conf_bin'] = pd.cut(df_results['confidence'], bins=5, 
                                   labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
    conf_acc = df_results.groupby('conf_bin')['correct'].mean()
    
    axes[1, 0].bar(range(len(conf_acc)), conf_acc.values, alpha=0.7, color='orange')
    axes[1, 0].set_title('Accuracy by Confidence Level')
    axes[1, 0].set_xlabel('Confidence Bins')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_xticks(range(len(conf_acc)))
    axes[1, 0].set_xticklabels(conf_acc.index, rotation=45)
    
    # Confusion matrix
    true_labels_numeric = [r['true_label_name'] for r in lime_results]
    pred_labels_numeric = [r['predicted_class'] for r in lime_results]
    label_names = ['entailment', 'neutral', 'contradiction']
    cm = confusion_matrix(true_labels_numeric, pred_labels_numeric, labels=label_names)
    
    im = axes[1, 1].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[1, 1].set_title('Confusion Matrix')
    tick_marks = np.arange(len(label_names))
    axes[1, 1].set_xticks(tick_marks)
    axes[1, 1].set_yticks(tick_marks)
    axes[1, 1].set_xticklabels(label_names, rotation=45)
    axes[1, 1].set_yticklabels(label_names)
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        axes[1, 1].text(j, i, format(cm[i, j], 'd'),
                       ha="center", va="center",
                       color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "model_performance_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. LIME Attribution Analysis
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Attribution score distribution
    all_scores = []
    for result in lime_results:
        attributions = result.get('lime_attributions_filtered', result['lime_attributions'])
        scores = [abs(score) for _, score in attributions]
        all_scores.extend(scores)
    
    axes[0, 0].hist(all_scores, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[0, 0].set_title('LIME Attribution Score Distribution')
    axes[0, 0].set_xlabel('Absolute Attribution Score')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].axvline(np.mean(all_scores), color='red', linestyle='--', 
                       label=f'Mean: {np.mean(all_scores):.3f}')
    axes[0, 0].legend()
    
    # LIME score distribution
    lime_scores = [r['lime_score'] for r in lime_results]
    axes[0, 1].hist(lime_scores, bins=20, alpha=0.7, color='coral', edgecolor='black')
    axes[0, 1].set_title('LIME Score Distribution')
    axes[0, 1].set_xlabel('LIME Score')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].axvline(np.mean(lime_scores), color='red', linestyle='--',
                       label=f'Mean: {np.mean(lime_scores):.3f}')
    axes[0, 1].legend()
    
    # Confidence vs Faithfulness scatter (if metrics available)
    if metrics_results:
        # Create mapping from id to metrics
        metrics_dict = {m['id']: m for m in metrics_results}
        
        plot_data = []
        for result in lime_results:
            if result['id'] in metrics_dict:
                plot_data.append({
                    'confidence': result['confidence'],
                    'faithfulness': metrics_dict[result['id']]['faithfulness']
                })
        
        if plot_data:
            conf_vals = [d['confidence'] for d in plot_data]
            faith_vals = [d['faithfulness'] for d in plot_data]
            
            axes[1, 0].scatter(conf_vals, faith_vals, alpha=0.6, color='green')
            axes[1, 0].set_title('Confidence vs Faithfulness')
            axes[1, 0].set_xlabel('Model Confidence')
            axes[1, 0].set_ylabel('Faithfulness Score')
            
            # Add correlation
            correlation = np.corrcoef(conf_vals, faith_vals)[0, 1]
            axes[1, 0].text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
                           transform=axes[1, 0].transAxes, fontsize=12,
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))
    
    # Top contributing words
    word_scores = defaultdict(list)
    for result in lime_results:
        attributions = result.get('lime_attributions_filtered', result['lime_attributions'])
        for word, score in attributions:
            word_scores[word.lower()].append(abs(score))
    
    # Get top 15 words by average absolute score
    avg_word_scores = {word: np.mean(scores) for word, scores in word_scores.items()}
    top_words = sorted(avg_word_scores.items(), key=lambda x: x[1], reverse=True)[:15]
    
    words, scores = zip(*top_words)
    axes[1, 1].barh(range(len(words)), scores, alpha=0.7, color='teal')
    axes[1, 1].set_yticks(range(len(words)))
    axes[1, 1].set_yticklabels(words)
    axes[1, 1].set_title('Top 15 Contributing Words')
    axes[1, 1].set_xlabel('Average Absolute Attribution Score')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "lime_attribution_analysis.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Metrics Analysis (if available)
    if metrics_results:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        metrics_df = pd.DataFrame(metrics_results)
        
        # Metrics comparison
        metric_means = [metrics_df['faithfulness'].mean(), 
                       metrics_df['sufficiency'].mean(),
                       metrics_df['comprehensiveness'].mean()]
        metric_names = ['Faithfulness', 'Sufficiency', 'Comprehensiveness']
        
        bars = axes[0].bar(metric_names, metric_means, alpha=0.7, 
                          color=['red', 'blue', 'green'])
        axes[0].set_title('Average Attribution Metrics')
        axes[0].set_ylabel('Score')
        axes[0].tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for bar, value in zip(bars, metric_means):
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{value:.3f}', ha='center', va='bottom')
        
        # Metrics correlation
        correlation_matrix = metrics_df[['faithfulness', 'sufficiency', 'comprehensiveness']].corr()
        
        im = axes[1].imshow(correlation_matrix, cmap='RdYlBu', vmin=-1, vmax=1)
        axes[1].set_title('Metrics Correlation Matrix')
        axes[1].set_xticks(range(len(correlation_matrix.columns)))
        axes[1].set_yticks(range(len(correlation_matrix.columns)))
        axes[1].set_xticklabels(correlation_matrix.columns, rotation=45)
        axes[1].set_yticklabels(correlation_matrix.columns)
        
        # Add correlation values
        for i in range(len(correlation_matrix.columns)):
            for j in range(len(correlation_matrix.columns)):
                axes[1].text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                            ha='center', va='center', color='white', fontweight='bold')
        
        plt.colorbar(im, ax=axes[1])
        plt.tight_layout()
        plt.savefig(os.path.join(plots_dir, "metrics_analysis.png"), dpi=300, bbox_inches='tight')
        plt.close()

    # Reliability (raw)
    plot_reliability_diagram(lime_results, dirs)

    # Optional overlay if calibration exists and classifier provided
    cal_path = os.path.join(dirs['LIME_OUTPUT_DIR'], 'calibration.json')
    if classifier is not None and file_nonempty(cal_path):
        cal = read_json(cal_path)
        if isinstance(cal, dict) and "temperature" in cal:
            plot_reliability_diagram_with_temp(
                lime_results, dirs, classifier=classifier, temperature=float(cal["temperature"])
            )

    # Save metadata about the plots
    write_json(stamp, {"schema_version": SCHEMA_VERSION,
                       "meta":{"source":"lime_filtered","source_hash": src_hash}})
    
    print(f"✅ All plots saved to: {plots_dir}")

In [35]:
def export_summary_csv(lime_results, metrics_results, dirs):
    outp = os.path.join(dirs['LIME_OUTPUT_DIR'], "summary.csv")
    m = {r["id"]: r for r in metrics_results} if metrics_results else {}
    rows = []
    for r in lime_results:
        mid = r["id"]
        rows.append({
            "id": mid,
            "true": r["true_label_name"],
            "pred": r["predicted_class"],
            "conf": r["confidence"],
            "lime_score": r.get("lime_score", np.nan),
            "lime_runtime_sec": r.get("lime_runtime_sec", np.nan),
            "faithfulness": m.get(mid, {}).get("faithfulness", np.nan),
            "sufficiency": m.get(mid, {}).get("sufficiency", np.nan),
            "comprehensiveness": m.get(mid, {}).get("comprehensiveness", np.nan),
        })
    df = pd.DataFrame(rows)
    # df: your DataFrame with columns: id, TRUE, pred, conf, lime_score, lime_runtime_sec, faithfulness, sufficiency, comprehensiveness, ...
    for col in ["faithfulness", "sufficiency", "comprehensiveness"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
    df.to_csv(outp, index=False, encoding="utf-8")
    print(f"📄 Exported CSV: {outp}")

## MAIN EXECUTION PIPELINE


In [36]:
def main(num_examples=50, lime_num_samples=1000, chunk_size=64, force_rebuild=False, subset_n=None, run_name=None):
    """Enhanced main pipeline with all requested features"""
    print("🎯 Starting Enhanced RoBERTa-MNLI + LIME Pipeline for SNLI")
    print("🔧 New features: reproducibility, sanity checks, error analysis, stability sweep, comprehensive plots")
    
    # Set reproducible seeds
    set_all_seeds(42)
    
    # Clear GPU memory
    if torch.cuda.is_available():
        print("🔒 Clearing GPU memory...")
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("✅ GPU memory cleared.")
    else:
        print("❗ No GPU available; skipping memory cleanup.")
    
    try:
        # Derive a sensible run_name if not provided
        if run_name is None and subset_n is not None:
            run_name = f"smoke-{int(subset_n)}"

        # Step 1: Setup
        dirs = setup_directories(run_name=run_name)
        print("✅ Directories setup complete")
        
        # Save run configuration
        config = save_run_config(dirs, num_samples=num_examples, 
                                lime_num_samples=lime_num_samples, chunk_size=chunk_size)
        
        # Step 2: Load and sample data
        snli_data = load_snli_dataset_fixed(dirs)
        snli_sample = sample_snli_dataset_fixed(snli_data, dirs, num_samples=num_examples)
        processed_snli = preprocess_snli_for_roberta(snli_sample, dirs)

        if subset_n is not None:
            processed_snli = sorted(processed_snli, key=lambda r: int(r["id"]))[:int(subset_n)]
            num_examples = min(num_examples, len(processed_snli))
            print(f"[Subset mode] Using first {len(processed_snli)} examples")

        
        # Step 3: Load RoBERTa model with unit tests
        classifier = RoBERTaMNLIClassifier("roberta-large-mnli", use_fp16=True)
        
        # Step 4: Generate LIME explanations
        lime_results = generate_lime_explanations(classifier, processed_snli, dirs, 
                                                 num_examples=num_examples,
                                                 lime_num_samples=lime_num_samples,
                                                 chunk_size=chunk_size,
                                                 force_rebuild=force_rebuild)
        
        # Step 5: Filter stopwords
        filtered_results = filter_stopwords_from_lime(lime_results, dirs)
        
        # Step 6: Evaluate metrics
        metrics_results = compute_evaluation_metrics(classifier, filtered_results, k=3, dirs=dirs)
        export_summary_csv(filtered_results, metrics_results, dirs)

        # --- Step 7: Calibration (ECE + Temperature scaling) ---
        compute_and_save_calibration(classifier, processed_snli, dirs, n_bins=15, val_fraction=0.2)
        
        # Step 8: Sanity checks
        sanity_results = perform_sanity_checks(classifier, filtered_results, dirs)
        
        # Step 9: Error analysis
        error_results = error_analysis(filtered_results, dirs)
        
        # Step 10: LIME stability sweep
        stability_results = lime_stability_sweep(classifier, processed_snli, dirs)
        
        # Step 11: Enhanced analysis with plots
        final_results = analyze_results_enhanced(filtered_results, metrics_results, dirs)
        
        # Backfill all_probabilities for both files if missing
        raw_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_roberta.json")
        filt_path = os.path.join(dirs['LIME_OUTPUT_DIR'], "lime_explanations_filtered.json")
        backfill_all_probabilities_inplace(raw_path,  classifier, schema_version=SCHEMA_VERSION)
        backfill_all_probabilities_inplace(filt_path, classifier, schema_version=SCHEMA_VERSION)

        # Step 12: Create comprehensive visualizations
        create_comprehensive_plots(filtered_results, metrics_results, dirs, classifier=classifier)

        plot_runtime_distribution(filtered_results, dirs)
        plot_per_class_top_words(filtered_results, dirs, by="predicted_class", top_n=15)
        
        # Step 13: Generate report
        generate_report(
            filtered_results, 
            metrics_results,
            dirs, 
            config, 
            sanity_results=sanity_results, 
            error_results=error_results, 
            stability_results=stability_results
        )
        
        print("\n🎉 Enhanced pipeline completed successfully!")
        print("📁 Generated files:")
        print(f"   📊 Results: {dirs['LIME_OUTPUT_DIR']}/complete_evaluation_results.json")
        print(f"   🔧 Config: {dirs['LIME_OUTPUT_DIR']}/run_config.json")
        print(f"   🔍 Sanity: {dirs['LIME_OUTPUT_DIR']}/sanity_check_results.json")
        print(f"   ❌ Errors: {dirs['LIME_OUTPUT_DIR']}/error_analysis.json")
        print(f"   🔬 Stability: {dirs['LIME_OUTPUT_DIR']}/lime_stability_sweep.json")
        print(f"   📝 Report: {dirs['LIME_OUTPUT_DIR']}/SNLI_LIME_report.md")
        print(f"   📈 Plots: {dirs['LIME_OUTPUT_DIR']}/plots/")
        
        return final_results
        
    except Exception as e:
        print(f"❌ Enhanced pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # Run with default parameters (can be called with arguments)
    results = main(num_examples=1000, lime_num_samples=700, chunk_size=64, force_rebuild=False)

🎯 Starting Enhanced RoBERTa-MNLI + LIME Pipeline for SNLI
🔧 New features: reproducibility, sanity checks, error analysis, stability sweep, comprehensive plots
✅ All seeds set to 42
🔒 Clearing GPU memory...
✅ GPU memory cleared.
📂 RUN_ROOT: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data
✅ Directories setup complete
✅ Run config saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\run_config.json
📥 Loading SNLI dataset...
Loading from local cache...
✅ SNLI dataset loaded: 550152 training examples
Sample: {'premise': 'A person on a horse jumps over a broken down airplane.', 'hypothesis': 'A person is training his horse for a competition.', 'label': 1}
⏩ Found existing SNLI sample at c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\sampled_snli_data\snli_sample.json; loading…
✅ Already have 1000 samples (>= 1000); reusing.
🔄 Preprocessing SNLI data for RoBERTa-MNLI...
✅ Saved 1000 processed examples to: c:\Users\Work\OneDrive\Desktop

LIME Explanations:   1%|          | 10/1000 [04:43<8:08:26, 29.60s/it]

💾 Checkpoint: saved 10/1000 items


LIME Explanations:   2%|▏         | 20/1000 [10:32<8:39:24, 31.80s/it] 

💾 Checkpoint: saved 20/1000 items


...

💾 Checkpoint: saved 850 metrics items to c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\metrics_partial.json


Computing metrics: 100%|██████████| 1000/1000 [03:29<00:00,  4.76it/s]


📄 Exported CSV: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\summary.csv
🔍 Performing sanity checks...
Found 861 correct, 139 incorrect predictions
🔀 Running shuffle sanity test...
✅ Sanity check results saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\sanity_check_results.json
🔍 Performing error analysis...
✅ Error analysis saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\error_analysis.json
🔬 Running LIME stability mini-sweep...


Stability sweep: 100%|██████████| 20/20 [11:29<00:00, 34.49s/ex, ns=1000, kw=default, ex 10/10]


✅ Stability sweep results saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\lime_stability_sweep.json
📈 Enhanced results analysis...

🎯 FINAL EVALUATION RESULTS:
   Model Accuracy: 0.861
   Average Confidence: 0.934
   Average Faithfulness: 0.296 ± 0.356
   Average Sufficiency: 0.693 ± 0.321
   Average Comprehensiveness: 0.296 ± 0.356
✅ Complete results saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\complete_evaluation_results.json
📊 Creating comprehensive visualization plots...
📈 Plotting reliability diagram...
✅ All plots saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\plots
📈 Plotting LIME runtime distribution...
📈 Plotting top words per class...
📝 Generating SNLI LIME report...
🔄 Computing bootstrap confidence intervals...
✅ Bootstrap confidence intervals computed
✅ Report saved to: c:\Users\Work\OneDrive\Desktop\ChatGPT\Research\project\data\lime_outputs\SNLI_LIME_report.md

